# Analize e tratamentos de dados


## 1 Primeiros contatos com os dados brutos

In [3]:
# Importação das bibliotecas que vamos trabalhar
import pandas as pd
import numpy as np

In [4]:
# Carregando o banco e mostrando algumas informações básicas
df = pd.read_csv('base_saude_1500_linhas.csv')

# 1 - Informações do banco (tipos de dados e contagem de não nulos)
print("1 - Informações do banco\n")
df.info()
print("-------------------------------------------------------------------------------")

# 2 - Descrição estatística do banco, com quantidade de dados preenchidos por coluna, médias e quartis
print("\n\n2 - Descrição do banco, quantidade de dados preenchidos por coluna, médias e quartis\n\n", df.describe())
print("-------------------------------------------------------------------------------")

# 3 - Visualização das primeiras linhas da tabela
print("\n\n3 - Dados no início da tabela\n\n", df.head())

1 - Informações do banco

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id_paciente       1500 non-null   int64  
 1   nome              1500 non-null   str    
 2   idade             1464 non-null   float64
 3   sexo              1500 non-null   str    
 4   cidade            1500 non-null   str    
 5   estado            1500 non-null   str    
 6   peso_kg           1500 non-null   str    
 7   altura_m          1500 non-null   float64
 8   imc               1500 non-null   float64
 9   pressao_arterial  1209 non-null   str    
 10  glicemia          763 non-null    float64
 11  colesterol        792 non-null    float64
 12  diagnostico       1256 non-null   str    
dtypes: float64(5), int64(1), str(7)
memory usage: 152.5 KB
-------------------------------------------------------------------------------


2 - Descrição do banco, quantid

In [ ]:
# Uma amostra de 10 linhas aleatórias do banco
df.sample(10)

In [ ]:
# O topo do banco, as primeiras linhas
df.head()

In [ ]:
# O final do banco, as últimas linhas
df.tail()

In [ ]:
# Descrição estatística geral do DataFrame
df.describe()

In [ ]:
# Resumo estatístico
df.iloc[:, 5:].describe()

In [ ]:
# Contagem de linhas duplicadas
df.duplicated().sum()

In [ ]:
# Contagem de dados nulos ou vazios
df.isnull().sum()

In [ ]:
# Porcentagem dos dados nulos
(df.isnull().sum()/len(df))*100

## 2 Tratamento dos dados afim de normatiza e facilitar o trabalho

In [ ]:
# Primeiro passo: fazer uma cópia para comparação posterior e análise das alterações
df_clean = df.copy()

In [ ]:
# Verifiquei as possíveis variações na coluna 'sexo'
valores_unicos = df['sexo'].unique()
print(valores_unicos)

In [ ]:
# Normatização dos valores da coluna 'sexo'
df_clean['sexo'] = df_clean['sexo'].replace({
    'm': 'Masculino',
    'M': 'Masculino',
    'F': 'Feminino',
    'f': 'Feminino'
})

In [ ]:
# Estabelecendo um separador padrão
df_clean['pressao_arterial'] = df_clean['pressao_arterial'].replace('/', "x", regex=True)

In [ ]:
# Normatizando os caracteres, na coluna 'nome' e na 'cidade' só a primeira letra maiúscula e no 'estado' tudo maiúsculo
df_clean['nome'] = df_clean['nome'].str.title()
df_clean['cidade'] = df_clean['cidade'].str.title()
df_clean['estado'] = df_clean['estado'].str.upper()

In [ ]:
# Adequando a coluna 'peso_kg', após a correção convertemos em número novamente
df_clean['peso_kg'] = df_clean['peso_kg'].astype(str)
df_clean['peso_kg'] = df_clean['peso_kg'].str.replace(',', '.')
df_clean['peso_kg'] = pd.to_numeric(df_clean['peso_kg'], errors='coerce')

In [ ]:
# Normatizando o padrão de medida da coluna 'altura_m', após a correção convertemos em número novamente
df_clean['altura_m'] = df_clean['altura_m'].astype(str)
df_clean['altura_m'] = df_clean['altura_m'].str.replace(',', '.')
df_clean['altura_m'] = pd.to_numeric(df_clean['altura_m'], errors='coerce')

In [ ]:
# Criei uma nova coluna com uma fórmula de IMC para validar a coluna já existente
df_clean['imc_calculado'] = (df_clean['peso_kg'] / (df_clean['altura_m']**2)).round(1)

In [ ]:
# Aqui mostro apenas as duas colunas junto com o nome apenas para diferenciar e provar que os dados são reais
df_clean[['nome', 'imc', 'imc_calculado']].head()

## 3 Limpeza dos dados vamos trabalhar os campos vazios


In [ ]:
# Para fins didáticos e controle vou fazer mais uma cópia do DataFrame
df_sem_nulos = df_clean.copy()

In [ ]:
# Pela quantidade baixa de linhas faltantes, vou apagar essas linhas sem idade
df_sem_nulos['idade'].isna().sum()

In [ ]:
# Apaga as linhas onde APENAS a coluna 'idade' estiver vazia
df_sem_nulos = df_clean.dropna(subset=['idade'])

In [ ]:
# Também encontrei idades igual a zero
idade_zero = df_sem_nulos[df_sem_nulos['idade'] == 0]
qto = len(idade_zero)
print(f"Quantidade de linhas com idade igual a zero: {qto}")
print(idade_zero)

In [ ]:
# Mantém no df_sem_nulos apenas as linhas onde a idade é diferente de 0
df_sem_nulos = df_sem_nulos[df_clean['idade'] != 0]

# Organiza os números do índice novamente (0, 1, 2, 3...)
df_sem_nulos = df_sem_nulos.reset_index(drop=True)

In [ ]:
# Transformando idade em inteiro
# 1. Arredonda a idade para cima (ex: 32.1 vira 33.0)
df_sem_nulos['idade'] = np.ceil(df_sem_nulos['idade'])

# 2. Transforma a coluna no tipo Inteiro (remove o ".0" do final)
df_sem_nulos['idade'] = df_sem_nulos['idade'].astype(int)

# 3. Exibe as primeiras linhas para conferir o resultado
df_sem_nulos[['nome', 'idade']].head()

In [ ]:
# A quantidade de linhas vazias na coluna 'diagnostico' é grande mas vou preenchê-las usando o 'imc_calculado'
df_clean['diagnostico'].isna().sum()

In [ ]:
# 1. Definimos as condições baseadas na coluna 'imc_calculado'
condicoes = [
    df_clean['imc_calculado'] < 18.5,
    (df_clean['imc_calculado'] >= 18.5) & (df_clean['imc_calculado'] < 25.0),
    (df_clean['imc_calculado'] >= 25.0) & (df_clean['imc_calculado'] < 30.0),
    df_clean['imc_calculado'] >= 30.0
]

# 2. Definimos os diagnósticos correspondentes a cada condição acima
resultados = ['Baixo peso', 'Saudável', 'Sobrepeso', 'Obesidade']

# 3. Criamos uma coluna temporária com os diagnósticos calculados por IMC
diagnostico_por_imc = np.select(condicoes, resultados, default='Não Identificado')

# 4. Preenchemos na coluna original APENAS onde o valor atual for nulo (NaN)
df_sem_nulos['diagnostico'] = df_clean['diagnostico'].fillna(pd.Series(diagnostico_por_imc, index=df_clean.index))

# 5. Conferindo se os nulos da coluna desapareceram
print("Quantidade de nulos restantes em diagnóstico:", df_sem_nulos['diagnostico'].isna().sum())

In [ ]:
# Apaguei apenas 45 linhas do meu DataFrame inicial, isso representa 3% da minha base
print(df_sem_nulos.info())
print(df_sem_nulos.describe())
print(df_sem_nulos.head())

## 4 Elaboração de graficicos e analize correlação entre os dados

#### Calculando a correlação de Pearson entre 'imc_calculado' e 'glicemia'

In [ ]:
correlation = df_sem_nulos['imc_calculado'].corr(df_sem_nulos['glicemia'])
print(f"Correlação de Pearson entre 'imc_calculado' e 'glicemia': {correlation:.2f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.scatterplot(x='imc_calculado', y='glicemia', data=df_sem_nulos)
plt.title('Gráfico de Dispersão: IMC Calculado vs Glicemia')
plt.xlabel('IMC Calculado')
plt.ylabel('Glicemia')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

#### Análise de Correlação entre IMC Calculado e Colesterol

In [ ]:
# Calculando a correlação de Pearson entre 'imc_calculado' e 'colesterol'
correlation_cholesterol = df_sem_nulos['imc_calculado'].corr(df_sem_nulos['colesterol'])
print(f"Correlação de Pearson entre 'imc_calculado' e 'colesterol': {correlation_cholesterol:.2f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.scatterplot(x='imc_calculado', y='colesterol', data=df_sem_nulos)
plt.title('Gráfico de Dispersão: IMC Calculado vs Colesterol')
plt.xlabel('IMC Calculado')
plt.ylabel('Colesterol')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

#### Comparação das Correlações

In [ ]:
print(f"Correlação de 'imc_calculado' com 'glicemia': {correlation:.2f}")
print(f"Correlação de 'imc_calculado' com 'colesterol': {correlation_cholesterol:.2f}")

print("\nObservação: Ambas as correlações são muito próximas de zero, o que sugere uma relação linear muito fraca ou inexistente entre 'imc_calculado' e 'glicemia', e entre 'imc_calculado' e 'colesterol' nos seus dados.")

#### Heatmap de Correlação para Variáveis Numéricas

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Selecionando apenas as colunas numéricas do DataFrame
# df_sem_nulos.select_dtypes(include=[np.number]) inclui todas as colunas numéricas
numerical_df = df_sem_nulos.select_dtypes(include=[np.number])

# Calculando a matriz de correlação
correlation_matrix = numerical_df.corr()

# Criando o heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Heatmap de Correlação das Variáveis Numéricas')
plt.show()

#### Análise Comparativa por Idade e Sexo

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Criando faixas etárias para a coluna 'idade'
bins = [0, 12, 18, 35, 60, 95]
labels = ['Criança', 'Adolescente', 'Adulto Jovem', 'Adulto', 'Idoso']
df_sem_nulos['faixa_etaria'] = pd.cut(df_sem_nulos['idade'], bins=bins, labels=labels, right=False)

# Exibindo as primeiras linhas com a nova coluna
print(df_sem_nulos[['idade', 'faixa_etaria']].head())
print(df_sem_nulos['faixa_etaria'].value_counts().sort_index())

#### Glicemia Média por Faixa Etária e Sexo

In [ ]:
# Glicemia média por faixa etária e sexo
plt.figure(figsize=(12, 7))
sns.barplot(x='faixa_etaria', y='glicemia', hue='sexo', data=df_sem_nulos, palette='viridis', errorbar=None)
plt.title('Glicemia Média por Faixa Etária e Sexo')
plt.xlabel('Faixa Etária')
plt.ylabel('Glicemia Média')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

#### Colesterol Médio por Faixa Etária e Sexo

In [ ]:
# Colesterol médio por faixa etária e sexo
plt.figure(figsize=(12, 7))
sns.barplot(x='faixa_etaria', y='colesterol', hue='sexo', data=df_sem_nulos, palette='magma', errorbar=None)
plt.title('Colesterol Médio por Faixa Etária e Sexo')
plt.xlabel('Faixa Etária')
plt.ylabel('Colesterol Médio')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

#### Distribuição de Diagnóstico por Faixa Etária e Sexo

In [ ]:
# Distribuição de diagnóstico por faixa etária e sexo
plt.figure(figsize=(14, 8))
sns.countplot(x='faixa_etaria', hue='diagnostico', data=df_sem_nulos, palette='tab10')
plt.title('Distribuição de Diagnóstico por Faixa Etária')
plt.xlabel('Faixa Etária')
plt.ylabel('Contagem')
plt.legend(title='Diagnóstico')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

plt.figure(figsize=(14, 8))
sns.countplot(x='sexo', hue='diagnostico', data=df_sem_nulos, palette='tab10')
plt.title('Distribuição de Diagnóstico por Sexo')
plt.xlabel('Sexo')
plt.ylabel('Contagem')
plt.legend(title='Diagnóstico')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
df.to_excel('Dados_brutos.xlsx', index=False)
df_sem_nulos.to_excel('Dados_limpos.xlsx', index=False)